In [1]:
# Install pysam
!pip install pysam


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 871.0 kB/s  0:00:09m0:00:0100:01


In [25]:
import pysam
from collections import Counter
import math

# =======================
# CONFIGURATION
# =======================
REF_FASTA = "/Users/leandro/Desktop/github/NGS-data/ONT/RHv2HDR_241125/alleles.fasta"
BAM = "/Users/leandro/Desktop/github/NGS-data/ONT/RHv2HDR_241125/HDR-only/aln.filtered.bam"

ref_name = "jPCR-RHv2"          # fasta header you want to inspect (change as needed)
cut_pos = 749            # 0-based coordinate on this reference
window = 50              # +/- bp window (adjustable)

# Insert the expected HDR sequence in the window (must match reference orientation)
expected_wt_seq  = "gctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGTGATGGCAGAGGAAAGGAAGCCCTGCTTCCTCCAGAGGGacgttactggc"
expected_hdr_seq = "gctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGGCAAGAGTTCCAGCCGGGCTATttacttttgtaaaactttatggtttgtg"   # <-- REPLACE with your HDR sequence

# =======================
# LOAD DATA
# =======================
bam = pysam.AlignmentFile(BAM, "rb")
ref = pysam.FastaFile(REF_FASTA)

ref_seq = ref.fetch(ref_name)
win_start = cut_pos - window
win_end = cut_pos + window + 1  # end-exclusive

ref_win = ref_seq[win_start:win_end]

# sanity check
print("Reference window:", ref_win)
print("Expected HDR    :", expected_hdr_seq)
print("Expected WT     :", expected_wt_seq)

# =======================
# HELPERS
# =======================

def get_read_seq_in_window(read, ref_name, start, end):
    """Extract the read sequence aligned to reference interval [start, end)."""
    ref_positions = read.get_reference_positions(full_length=False)
    read_seq = read.query_sequence

    seq_out = []
    for qpos, rpos in read.get_aligned_pairs(matches_only=False):
        if rpos is None:  # insertion or softclip; skip for HDR classification
            continue
        if start <= rpos < end:
            if qpos is None:
                # deletion relative to reference → return gap placeholder
                seq_out.append("-")
            else:
                seq_out.append(read_seq[qpos])
    return "".join(seq_out)


def read_has_indel(read, cut_pos, window):
    """Check if a read has an indel overlapping the cut window."""
    ref_pos = read.reference_start
    for op, length in read.cigartuples:
        if op in (0, 7, 8):  # M, =, X
            ref_pos += length
        elif op == 2:  # deletion
            if (ref_pos < cut_pos + window) and (ref_pos + length > cut_pos - window):
                return True
            ref_pos += length
        elif op == 1:  # insertion
            if (ref_pos >= cut_pos - window) and (ref_pos <= cut_pos + window):
                return True
    return False


# =======================
# CLASSIFICATION
# =======================
counts = Counter()
total = 0

for read in bam.fetch(ref_name):
    if read.is_unmapped or read.is_secondary or read.is_supplementary:
        continue
    if read.mapping_quality < 20:
        continue
    
    a, b = read.reference_start, read.reference_end
    if a is None or b is None:
        continue
    
    # Must overlap the window
    if not (a <= win_end and b >= win_start):
        continue
    
    total += 1

    # NHEJ: indel near cut
    if read_has_indel(read, cut_pos, window):
        counts["NHEJ"] += 1
        continue
    
    # Extract sequence in window
    rseq = get_read_seq_in_window(read, ref_name, win_start, win_end)

    if len(rseq.replace("-", "")) < (2 * window): 
        # Too gappy, skip
        counts["Other"] += 1
        continue

    # HDR: exact match to expected donor sequence
    if rseq == expected_hdr_seq:
        counts["HDR"] += 1
        continue

    # WT: matches WT motif
    if rseq == expected_wt_seq:
        counts["WT"] += 1
        continue
    
    # Otherwise ambiguous
    counts["Other"] += 1


# =======================
# SUMMARY
# =======================
print("\n=== RESULTS ===")
for k, v in counts.items():
    print(f"{k}: {v}")
print("Total reads covering window:", total)

# HDR efficiency
if total > 0:
    hdr_p = counts["HDR"] / total
    print(f"HDR % = {hdr_p*100:.2f}%")
else:
    print("HDR % = 0.00%")

# Wilson 95% CI
z = 1.96
n = total
p = counts["HDR"] / n if n>0 else 0

if n > 0:
    center = (p + z*z/(2*n)) / (1 + z*z/n)
    half = (z * math.sqrt((p*(1-p)/n) + z*z/(4*n*n))) / (1 + z*z/n)
    lower = max(0, center - half)
    upper = min(1, center + half)
    print(f"95% CI: {100*lower:.2f}% – {100*upper:.2f}%")


Reference window: agctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGGCAAGAGTTCCAGCCGGGCTATttacttttgtaaaactttatggtttgtg
Expected HDR    : gctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGGCAAGAGTTCCAGCCGGGCTATttacttttgtaaaactttatggtttgtg
Expected WT     : gctatCTATATATAGCTATCTATGTCTCCTTGGGGCCCAGACTGAGCACGTGATGGCAGAGGAAAGGAAGCCCTGCTTCCTCCAGAGGGacgttactggc

=== RESULTS ===
Other: 1094
NHEJ: 361
Total reads covering window: 1455
HDR % = 0.00%
95% CI: 0.00% – 0.26%


In [26]:
import difflib
from collections import Counter
import math

# ---- CONFIG ---- (keep existing BAM/ref/window/cut_pos/expected_* definitions)
FUZZY_THRESHOLD = 0.9   # sequence identity threshold to call HDR (0-1). Raise to 0.95 for stricter.

def get_read_seq_in_window_including_insertions(read, win_start, win_end):
    """
    Reconstruct the read sequence aligned to reference interval [win_start, win_end),
    *including* inserted bases (qpos present, rpos None) that fall inside the window.
    We build a string by scanning aligned pairs in order.
    Returns a string where deletions relative to the read are '-' and insertions are inserted bases.
    """
    seq = []
    read_seq = read.query_sequence or ""
    # get_aligned_pairs returns list of (qpos, rpos) in read order
    for qpos, rpos in read.get_aligned_pairs(matches_only=False):
        # If reference position is within window, keep the read base (or '-' for deletions)
        if rpos is not None and win_start <= rpos < win_end:
            if qpos is None:
                seq.append("-")          # deletion relative to read
            else:
                seq.append(read_seq[qpos])
        # Also include insertions: rpos is None, but the insertion point is between reference positions.
        # We include an insertion if either the previous or next reference position (neighbour) is inside the window.
        elif rpos is None and qpos is not None:
            # find neighbour reference positions (context) by peeking surrounding aligned_pairs is expensive;
            # simpler heuristic: include insertion if there exists any aligned pair with rpos inside window
            # near this insertion — we will include if insertion occurs between two aligned bases and either neighbor is in window.
            # We'll include all insertions and filter later by whether they fall adjacent to window by index.
            # For practical amplicon alignments, insertions near the cut are frequent; include them to allow HDR detection.
            seq.append(read_seq[qpos])
    return "".join(seq).upper()

def read_has_indel_nearby(read, cut_pos, window):
    """Same logic as before — detect D or I overlapping window."""
    ref_pos = read.reference_start
    for op, length in read.cigartuples or []:
        # CIGAR ops: 0=M,1=I,2=D,3=N,4=S,5=H,7=X,8==
        if op in (0, 7, 8):
            ref_pos += length
        elif op == 2:  # deletion consumes reference
            if (ref_pos < cut_pos + window) and (ref_pos + length > cut_pos - window):
                return True
            ref_pos += length
        elif op == 1:  # insertion (occurs at ref_pos)
            if (ref_pos >= cut_pos - window) and (ref_pos <= cut_pos + window):
                return True
            # ref_pos unchanged
        else:
            # softclip/hardclip: do not advance ref_pos
            pass
    return False

# ---- classification loop (replace your previous loop) ----
counts = Counter()
total = 0
debug = Counter()   # track reasons: fuzzy_match, exact_substring, too_short

expected_hdr_up = expected_hdr_seq.upper().replace("\n", "").replace(" ", "")
expected_wt_up  = expected_wt_seq.upper().replace("\n", "").replace(" ", "")
expected_len = len(expected_hdr_up)

for read in bam.fetch(ref_name):
    if read.is_unmapped or read.is_secondary or read.is_supplementary:
        continue
    if read.mapping_quality < 20:
        continue
    a, b = read.reference_start, read.reference_end
    if a is None or b is None:
        continue
    # window overlap test (win_start/win_end should be defined earlier)
    if not (a <= win_end and b >= win_start):
        continue
    total += 1

    # NHEJ by indel in window
    if read_has_indel_nearby(read, cut_pos, window):
        counts["NHEJ"] += 1
        continue

    # reconstruct read sequence in window (including insertions)
    rseq = get_read_seq_in_window_including_insertions(read, win_start, win_end)
    rseq_clean = rseq.replace("-", "")  # remove deletion placeholders for matching
    rseq_clean = rseq_clean.upper()

    # skip very short coverage
    if len(rseq_clean) < max( int(0.6 * expected_len), expected_len - 5 ):
        counts["Other"] += 1
        debug["too_short"] += 1
        continue

    # 1) Exact substring match (fast, tolerant to flanking)
    if expected_hdr_up in rseq_clean:
        counts["HDR"] += 1
        debug["exact_substring"] += 1
        continue

    # 2) Fuzzy matching using SequenceMatcher ratio
    ratio = difflib.SequenceMatcher(None, expected_hdr_up, rseq_clean).ratio()
    if ratio >= FUZZY_THRESHOLD:
        counts["HDR"] += 1
        debug["fuzzy_match"] += 1
        continue

    # 3) WT exact substring
    if expected_wt_up in rseq_clean:
        counts["WT"] += 1
        continue

    counts["Other"] += 1

# ---- summary (same as before) ----
print("\n=== RESULTS ===")
for k, v in counts.items():
    print(f"{k}: {v}")
print("Total reads covering window:", total)

if total > 0:
    hdr_p = counts["HDR"] / total
    print(f"HDR % = {hdr_p*100:.2f}%")
else:
    print("HDR % = 0.00%")

# Wilson 95% CI
z = 1.96
n = total
p = counts["HDR"] / n if n>0 else 0
if n > 0:
    center = (p + z*z/(2*n)) / (1 + z*z/n)
    half = (z * math.sqrt((p*(1-p)/n) + z*z/(4*n*n))) / (1 + z*z/n)
    lower = max(0, center - half)
    upper = min(1, center + half)
    print(f"95% CI: {100*lower:.2f}% – {100*upper:.2f}%")

print("\nDebug info:", dict(debug))



=== RESULTS ===
HDR: 1082
NHEJ: 361
Other: 12
Total reads covering window: 1455
HDR % = 74.36%
95% CI: 72.06% – 76.54%

Debug info: {'exact_substring': 1023, 'fuzzy_match': 59, 'too_short': 8}
